In [1]:
import pandas as pd
import geopandas as gpd
import os

import sys
sys.path.append('..')
from helpers import get_county2zone, calculate_intersection

In [2]:
# import io
# import zipfile
# import requests

# def download_tract_data(state_fips):
#     tract = f'tl_2022_{state_fips}_tract'
#     zip_file_url = f'https://www2.census.gov/geo/tiger/TIGER2022/TRACT/{tract}.zip'

#     for _ in range(5):
#         try:
#             response = requests.get(zip_file_url, stream=True)
#             response.raise_for_status()
#             break
#         except:
#             print(f"Got status code {response.status_code}. Retrying API call.")

#     out_dir = f'../data/shapefiles/census_tracts/{tract}'
#     os.makedirs(out_dir, exist_ok=True)
#     with zipfile.ZipFile(io.BytesIO(response.content)) as z:
#         z.extractall(out_dir)

county2zone = get_county2zone(2023)
county2zone['state_fips'] = county2zone['FIPS'].str.split('p').str[1].str[:2]
# for state_fips in county2zone['state_fips'].unique().tolist():
#     print(f"Downloading state {state_fips}...", end="")
#     download_tract_data(state_fips)
#     print(" done")

In [3]:
eia_zones = gpd.read_file('../data/shapefiles/bas_and_subbas')

In [4]:
ct_pops = pd.read_csv('../data/DECENNIALDHC2020.P1_2025-08-20T004859/DECENNIALDHC2020.P1-Data.csv')
ct_pops['GEOID'] = ct_pops['GEO_ID'].str.split('US').str[1]
ct_pops = ct_pops.rename(columns={'P1_001N': 'population'})
ct_pops = ct_pops[['GEOID', 'population']]

In [5]:
dfs = []
for state_fips in county2zone['state_fips'].unique().tolist():
    if state_fips == '09':
        continue

    print(f"Processing state {state_fips}...", end="")

    state_cts = (
        gpd.read_file(f'../data/shapefiles/census_tracts/tl_2022_{state_fips}_tract')
        .to_crs(eia_zones.crs)
    )
    assert len(state_cts.merge(ct_pops, on='GEOID')) == len(state_cts)
    state_cts = state_cts.merge(ct_pops, on='GEOID')
    state_cts['population_density'] = state_cts['population'].astype(float) / state_cts['ALAND']

    zone_ct_intersects = calculate_intersection(eia_zones, state_cts, ['EIAcode', 'GEOID'])
    zone_ct_intersects['population'] = (
        zone_ct_intersects['population_density'] * zone_ct_intersects['geometry'].area
    )
    zone_ct_intersects['FIPS'] = (
        'p' + zone_ct_intersects['STATEFP'].astype(str) + zone_ct_intersects['COUNTYFP'].astype(str)
    )
    zone_ct_intersects['coverage'] = (
        zone_ct_intersects['population'] / zone_ct_intersects.groupby('FIPS')['population'].transform('sum')
    )
    df = zone_ct_intersects.groupby(['FIPS', 'EIAcode'], as_index=False)['coverage'].sum()
    dfs.append(df)

    print(" done")

Processing state 53... done
Processing state 41... done
Processing state 06... done
Processing state 32... done
Processing state 16... done
Processing state 30... done
Processing state 56... done
Processing state 49... done
Processing state 04... done
Processing state 35... done
Processing state 46... done
Processing state 08... done
Processing state 38... done
Processing state 31... done
Processing state 27... done
Processing state 19... done
Processing state 55... done
Processing state 48... done
Processing state 40... done
Processing state 20... done
Processing state 29... done
Processing state 05... done
Processing state 22... done
Processing state 26... done
Processing state 17... done
Processing state 28... done
Processing state 01... done
Processing state 12... done
Processing state 47... done
Processing state 21... done
Processing state 13... done
Processing state 45... done
Processing state 37... done
Processing state 51... done
Processing state 18... done
Processing state 39.

In [6]:
county_zone_area_coverage = pd.concat(dfs, ignore_index=True)
for _, row in county2zone.loc[county2zone.state == 'CT'].iterrows():
    county_zone_area_coverage.loc[len(county_zone_area_coverage)] = [row['FIPS'], '4004', 1]

In [7]:
assert county2zone.loc[~county2zone.FIPS.isin(county_zone_area_coverage.FIPS)].empty

In [8]:
county_zone_area_coverage.to_csv('../data/county_zone_area_coverage.csv', index=False)